In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'

# Creator credentials (template registrar)
idp_name_creator = 'FakeCAS'
idp_username_creator = None
idp_password_creator = None

# Manager credentials (project admin)
idp_name_manager = 'FakeCAS'
idp_username_manager = None
idp_password_manager = None

# Executor credentials (workflow initiator)
idp_name_executor = 'FakeCAS'
idp_username_executor = None
idp_password_executor = None

# Template project (where creator registers the template)
template_project_name = None

# Execution project (where manager/executor run the workflow)
project_name = None

# Workflow settings
engine_name = None
workflow_zip_path = 'resources/creator-approval-workflow.zip'
workflow_name = None

# Test data
start_form_content = 'Test request content'

default_result_path = None
close_on_fail = False
transition_timeout = 30000
browser_type = 'chromium'
ignore_https_errors = False

In [ ]:
import tempfile

if idp_username_creator is None:
    idp_username_creator = input(prompt=f'Username for creator ({idp_name_creator})')
if idp_password_creator is None:
    idp_password_creator = getpass(prompt=f'Password for {idp_username_creator}@{idp_name_creator}')

if idp_username_manager is None:
    idp_username_manager = input(prompt=f'Username for manager ({idp_name_manager})')
if idp_password_manager is None:
    idp_password_manager = getpass(prompt=f'Password for {idp_username_manager}@{idp_name_manager}')

if idp_username_executor is None:
    idp_username_executor = input(prompt=f'Username for executor ({idp_name_executor})')
if idp_password_executor is None:
    idp_password_executor = getpass(prompt=f'Password for {idp_username_executor}@{idp_name_executor}')

if template_project_name is None:
    template_project_name = input(prompt='Template project name (for creator)')
if project_name is None:
    project_name = datetime.now().strftime('TEST-WORKFLOW-CREATOR-%Y%m%d%H%M')
if engine_name is None:
    engine_name = input(prompt='Workflow engine name')
if workflow_name is None:
    workflow_name = datetime.now().strftime('CREATOR-APPROVAL-%Y%m%d%H%M')

# ワークフローの実行（creator承認フロー）

- サブシステム名: アドオン
- ページ/アドオン: Workflow
- 機能分類: ワークフロー実行
- シナリオ名: 3者（executor, manager, creator）によるワークフロー実行と承認
- 用意するテストデータ: アカウント(creator: テンプレート登録者, manager: プロジェクト管理者, executor: 申請者)、テンプレート登録用プロジェクト、ワークフロー実行用プロジェクト
- 事前条件: エンジン登録が完了しており、テンプレート登録用プロジェクトがアップロード許可ノードIDに登録されていること
- 備考: 3ユーザーは必ず別々のアカウントを使用すること。creatorのタスク・通知はテンプレート登録プロジェクトに表示される。

In [ ]:
work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import asyncio
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

## 新規タブを開き、GRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    button = page.locator('//button[text() = "同意する"]')
    await button.scroll_into_view_if_needed()
    await button.click()
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step, new_page=True)

## creatorとしてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_creator, idp_username_creator, idp_password_creator, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧からテンプレート登録用プロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
template_project_url = None

async def _step(page):
    global template_project_url
    project_item = page.locator(f'//*[@data-test-dashboard-item-title and text() = "{template_project_name}"]')
    await project_item.scroll_into_view_if_needed()
    await project_item.click()
    title = page.locator('//span[@id = "nodeTitleEditable"]')
    await expect(title).to_be_visible(timeout=transition_timeout)
    await title.scroll_into_view_if_needed()
    template_project_url = page.url

await run_pw(_step)

## 「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    addons_link = page.locator('//a[text() = "アドオン"]')
    await addons_link.scroll_into_view_if_needed()
    await addons_link.click()
    header = page.locator('//h3[text() = "アドオンを選択"]')
    await expect(header).to_be_visible(timeout=transition_timeout)
    await header.scroll_into_view_if_needed()

await run_pw(_step)

## 「Workflow」の「有効にする」をクリックする

Workflowアドオンが有効化されること（すでに有効な場合はスキップ）

In [ ]:
async def _step(page):
    enable_button = page.locator('//div[@full_name = "Workflow"]//a[text() = "有効にする"]')
    if await enable_button.count():
        await enable_button.scroll_into_view_if_needed()
        await enable_button.click()
        confirm_button = page.locator('//button[@data-bb-handler = "confirm"]')
        await expect(confirm_button).to_be_visible(timeout=transition_timeout)
        await confirm_button.scroll_into_view_if_needed()
        await confirm_button.click()
    engine_select = page.locator('#workflow-engine-id')
    await expect(engine_select).to_be_visible(timeout=transition_timeout)
    await engine_select.scroll_into_view_if_needed()

await run_pw(_step)

## 「ワークフローエンジン」ドロップダウンからエンジンを選択する

指定したエンジンが選択されること

In [ ]:
async def _step(page):
    engine_select = page.locator('#workflow-engine-id')
    await engine_select.scroll_into_view_if_needed()
    option = page.locator(f'#workflow-engine-id option:has-text("{engine_name}")')
    value = await option.get_attribute('value')
    await engine_select.select_option(value=value)

await run_pw(_step)

## 「ワークフローZIP」にBPMNファイルを含むZIPをアップロードする

ZIPファイルが選択されること

In [ ]:
async def _step(page):
    zip_input = page.locator('#workflow-zip')
    await zip_input.scroll_into_view_if_needed()
    await zip_input.set_input_files(workflow_zip_path)

await run_pw(_step)

## 「名前」にワークフロー名を入力する

ワークフロー名が入力されること

In [ ]:
async def _step(page):
    name_input = page.locator('#workflow-label')
    await name_input.scroll_into_view_if_needed()
    await name_input.fill(workflow_name)

await run_pw(_step)

## 「公開範囲」ドロップダウンで「同じ機関のユーザー」を選択する

「同じ機関のユーザー」が選択されること

In [ ]:
async def _step(page):
    visibility_select = page.locator('#workflow-visibility')
    await visibility_select.scroll_into_view_if_needed()
    await visibility_select.select_option(value='institution')

await run_pw(_step)

## 「作成者トークン」ドロップダウンで「閲覧＆編集権限で使用」を選択する

「閲覧＆編集権限で使用」が選択されること

In [ ]:
async def _step(page):
    token_mode_select = page.locator('select[data-bind="value: form.creatorTokenMode"]')
    await token_mode_select.scroll_into_view_if_needed()
    await token_mode_select.select_option(value='readwrite')

await run_pw(_step)

## 「管理者トークン」ドロップダウンで「閲覧＆編集権限で使用」を選択する

「閲覧＆編集権限で使用」が選択されること

In [ ]:
async def _step(page):
    token_mode_select = page.locator('select[data-bind="value: form.managerTokenMode"]')
    await token_mode_select.scroll_into_view_if_needed()
    await token_mode_select.select_option(value='readwrite')

await run_pw(_step)

## 「ワークフローテンプレートを登録」をクリックする

「ワークフロー権限の付与」ダイアログが表示されること

In [ ]:
async def _step(page):
    register_button = page.locator('//button[.//span[contains(text(), "ワークフローテンプレートを登録")]]')
    await register_button.scroll_into_view_if_needed()
    await register_button.click()
    await expect(page.locator(".modal:visible").locator('//h4[text() = "ワークフロー権限の付与"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「権限を付与してワークフローテンプレートを登録」をクリックする

テンプレートが作成され、ローカルワークフロー一覧に表示されること

In [ ]:
async def _step(page):
    register_button = page.locator('//button[contains(text(), "権限を付与してワークフローテンプレートを登録")]')
    await register_button.click()
    template_row = page.locator(f'#localWorkflowsPanel').locator(f'//td//strong[text()="{workflow_name}"]')
    await expect(template_row).to_be_visible(timeout=transition_timeout)
    await template_row.scroll_into_view_if_needed()

await run_pw(_step)

## creatorをログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_creator, transition_timeout=transition_timeout)

await run_pw(_step)

## managerとしてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    await grdm.login(page, idp_name_manager, idp_username_manager, idp_password_manager, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクト一覧に指定されたタイトルのプロジェクトがない場合、プロジェクトを作成する

プロジェクト一覧に当該プロジェクト名が表示されていない場合、「新規プロジェクト作成」をクリックし、その名前を入力、「作成」をクリックする。

In [ ]:
async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)
    await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から対象プロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
project_url = None

async def _step(page):
    global project_url
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンを選択」のパネル内「Workflow」の行の「有効にする」をクリックする

「Workflowアドオン規約」のダイアログが表示されること

In [ ]:
async def _step(page):
    enable_button = page.locator('//div[@full_name = "Workflow"]//a[text() = "有効にする"]')
    if await enable_button.count():
        await enable_button.click()
        await expect(page.locator('//button[@data-bb-handler = "confirm"]')).to_be_visible(timeout=transition_timeout)
    else:
        print('Workflow addon is already enabled for this project')

await run_pw(_step)

## 「確認」をクリックする

「アドオンを構成」のパネル内に「Workflow」の行が追加されること

In [ ]:
async def _step(page):
    confirm_button = page.locator('//button[@data-bb-handler = "confirm"]')
    if await confirm_button.count():
        await confirm_button.click()
    await expect(page.locator('#activate-workflow-select')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ワークフローテンプレート」ドロップダウンからテンプレートを選択する

指定したテンプレートが選択されること

In [ ]:
async def _step(page):
    await page.locator('#activate-workflow-select').scroll_into_view_if_needed()
    option = page.locator(f'#activate-workflow-select option:has-text("{workflow_name}")')
    value = await option.get_attribute('value')
    await page.locator('#activate-workflow-select').select_option(value=value)

await run_pw(_step)

## 「有効化」をクリックする

「ワークフロー権限の付与」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('#activationsPanel').locator('//button[.//span[contains(text(), "有効化")]]').click()
    await expect(page.locator("#activateTokenPermissionModal").locator('//h4[contains(text(), "ワークフロー権限の付与")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「権限を付与して有効化」をクリックする

テンプレートが有効化され、有効化済みテンプレート一覧に表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[contains(text(), "権限を付与して有効化")]').click()
    await expect(page.locator(f'#activationsPanel').locator(f'//td//strong[text()="{workflow_name}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクト名をクリックしてプロジェクトダッシュボードに戻る

プロジェクトダッシュボードが表示され、ワークフローパネルにテンプレートリンクが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//a[contains(@class, "project-title") and normalize-space(text()) = "{project_name}"]').click()
    await expect(page.locator('#workflow-dashboard')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'#workflow-dashboard').locator(f'//a[contains(@class, "btn-primary") and contains(text(), "{workflow_name}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## managerをログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_manager, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクトURL/workflowにアクセスしexecutorとしてログインする

「許可が必要です」画面が表示され、「アクセス権をリクエスト」ボタンが表示されること

In [ ]:
async def _step(page):
    await page.goto(project_url.rstrip('/') + '/workflow')
    await grdm.login(page, idp_name_executor, idp_username_executor, idp_password_executor, transition_timeout=transition_timeout)
    await expect(page.locator('//button[text() = "アクセス権をリクエスト"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「アクセス権をリクエスト」をクリックする

「アクセス権がリクエストされました」ボタンが表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[text() = "アクセス権をリクエスト"]').click()
    await expect(page.locator('//button[text() = "アクセス権がリクエストされました"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## executorをログアウトしプロジェクトURLを開きmanagerとしてログインする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_executor, transition_timeout=transition_timeout)
    await page.goto(project_url)
    await grdm.login(page, idp_name_manager, idp_username_manager, idp_password_manager, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「メンバー」をクリックする

「アクセスのリクエスト」セクションが表示され、executorからのリクエストが表示されること

In [ ]:
async def _step(page):
    await page.locator('#projectSubnav').get_by_role('link', name='メンバー').click()
    await expect(page.locator('//h3[contains(text(), "アクセスのリクエスト")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('.request-accept-button')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 権限を「読込み/書込み」に変更し「追加」をクリックする

リクエストが承認され、「アクセスのリクエスト」セクションからexecutorの行が消えること

In [ ]:
async def _step(page):
    await page.locator('#manageAccessRequests .permissions select').select_option('write')
    await page.locator('.request-accept-button').click()
    await expect(page.locator('#manageAccessRequests .permissions select')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## managerをログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_manager, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクトURLを開きexecutorとしてログインする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    await page.goto(project_url)
    await grdm.login(page, idp_name_executor, idp_username_executor, idp_password_executor, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ワークフローパネルのテンプレートリンクをクリックする

ワークフロー開始フォームが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'#workflow-dashboard').locator(f'//a[contains(@class, "btn-primary") and contains(text(), "{workflow_name}")]').click()
    await expect(page.locator('#workflow-field-confirm_checkbox')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(3)

await run_pw(_step)

## 「内容を確認しました」をチェックし「申請内容」に値を入力する

入力内容が反映されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-field-confirm_checkbox').check()
    await page.locator('#workflow-field-request_content').fill(start_form_content)
    await expect(page.locator('#workflow-field-confirm_checkbox')).to_be_checked(timeout=transition_timeout)
    await expect(page.locator('#workflow-field-request_content')).to_have_value(start_form_content, timeout=transition_timeout)

await run_pw(_step)

## 「ワークフローを開始」をクリックする

ワークフローが開始され、成功メッセージが表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[@data-test-workflow-start-button]').click()
    await expect(page.locator('//*[@data-test-workflow-submit-success]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## executorをログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_executor, transition_timeout=transition_timeout)

await run_pw(_step)

## 10秒待ち、テンプレートプロジェクトURLを開きcreatorとしてログインし、コメントを確認する

コメントパネルに申請内容を含む通知コメントが表示されること（creatorへの開始通知）

In [ ]:
async def _step(page):
    await asyncio.sleep(10)
    await page.goto(template_project_url)
    await grdm.login(page, idp_name_creator, idp_username_creator, idp_password_creator, transition_timeout=transition_timeout)
    await expect(page.locator('.fa-comments-o')).to_be_visible(timeout=transition_timeout)
    await page.locator('.fa-comments-o').click()
    await expect(page.locator(f'//*[contains(text(), "{start_form_content}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## creatorをログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_creator, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクトURLを開きmanagerとしてログインする

プロジェクトダッシュボードに承認タスクの「開く」ボタンが表示されること

In [ ]:
async def _step(page):
    await page.goto(project_url)
    await grdm.login(page, idp_name_manager, idp_username_manager, idp_password_manager, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_be_visible(timeout=transition_timeout)
    await page.locator('#workflow-dashboard').locator('.fa-pencil').scroll_into_view_if_needed()

await run_pw(_step)

## 承認タスクの「開く」をクリックする

審査フォームダイアログが表示され、executorが入力した申請内容が表示されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-dashboard').locator('.fa-pencil').click()
    await expect(page.locator('#workflow-field-approval_result')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'//*[contains(text(), "{start_form_content}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 審査フォームで「再提出」を選択し「コメント」を入力する

入力内容が反映されること

In [ ]:
resubmit_comment = 'Please revise and resubmit'

async def _step(page):
    option = page.locator('#workflow-field-approval_result option:has-text("再提出")')
    value = await option.get_attribute('value')
    await page.locator('#workflow-field-approval_result').select_option(value=value)
    await page.locator('#workflow-field-review_comment').fill(resubmit_comment)

await run_pw(_step)

## 「送信」をクリックする

タスクが送信され、managerにはタスクが表示されないこと（executorに割り当てられている）

In [ ]:
async def _step(page):
    await page.locator('//button[@data-test-task-dialog-submit]').click()
    await expect(page.locator('//button[@data-test-task-dialog-submit]')).not_to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//button[@data-test-workflow-task-open]')).to_have_count(0, timeout=transition_timeout)
    await asyncio.sleep(2)

await run_pw(_step)

## managerをログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_manager, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクトURLを開きexecutorとしてログインする

再申請タスクの「開く」ボタンが表示されること

In [ ]:
async def _step(page):
    await page.goto(project_url)
    await grdm.login(page, idp_name_executor, idp_username_executor, idp_password_executor, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## コメントボタンをクリックする

コメントパネルが開き、再提出理由を含む通知コメントが表示されること

In [ ]:
async def _step(page):
    await expect(page.locator('.fa-comments-o')).to_be_visible(timeout=transition_timeout)
    await page.locator('.fa-comments-o').click()
    await expect(page.locator(f'//*[contains(text(), "{resubmit_comment}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 再申請タスクの「開く」をクリックする

再申請フォームダイアログが表示されること

In [ ]:
async def _step(page):
    await page.reload()
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_be_visible(timeout=transition_timeout)
    await page.locator('#workflow-dashboard').locator('.fa-pencil').click()
    await expect(page.locator('#workflow-field-confirm_checkbox')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「申請内容」に再提出メッセージを入力し「内容を確認しました」をチェックして「送信」をクリックする

タスクが送信され、executorにはタスクが表示されないこと（managerに割り当てられている）

In [ ]:
resubmit_content = 'Revised and resubmitted'

async def _step(page):
    await page.locator('#workflow-field-request_content').fill(resubmit_content)
    await page.locator('#workflow-field-confirm_checkbox').check()
    await page.locator('//button[@data-test-task-dialog-submit]').click()
    await expect(page.locator('//button[@data-test-task-dialog-submit]')).not_to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//button[@data-test-workflow-task-open]')).to_have_count(0, timeout=transition_timeout)
    await asyncio.sleep(2)

await run_pw(_step)

## executorをログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_executor, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクトURLを開きmanagerとしてログインする

プロジェクトダッシュボードに承認タスクの「開く」ボタンが表示されること

In [ ]:
async def _step(page):
    await page.goto(project_url)
    await grdm.login(page, idp_name_manager, idp_username_manager, idp_password_manager, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 承認タスクの「開く」をクリックする

審査フォームダイアログが表示され、executorが再入力した申請内容が表示されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-dashboard').locator('.fa-pencil').click()
    await expect(page.locator('#workflow-field-approval_result')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'//*[contains(text(), "{resubmit_content}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 審査フォームで「承認」を選択し「コメント」を入力して「送信」をクリックする

タスクが送信され、managerにはタスクが表示されないこと（creatorに最終承認依頼）

In [ ]:
manager_approval_comment = 'Approved by manager'

async def _step(page):
    option = page.locator('#workflow-field-approval_result option:has-text("承認")')
    value = await option.get_attribute('value')
    await page.locator('#workflow-field-approval_result').select_option(value=value)
    await page.locator('#workflow-field-review_comment').fill(manager_approval_comment)
    await page.locator('//button[@data-test-task-dialog-submit]').click()
    await expect(page.locator('//button[@data-test-task-dialog-submit]')).not_to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//button[@data-test-workflow-task-open]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## managerをログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_manager, transition_timeout=transition_timeout)

await run_pw(_step)

## 10秒待ち、テンプレートプロジェクトURLを開きcreatorとしてログインする

プロジェクトダッシュボードに最終承認タスクの「開く」ボタンが表示されること

In [ ]:
async def _step(page):
    await asyncio.sleep(10)
    await page.goto(template_project_url)
    await grdm.login(page, idp_name_creator, idp_username_creator, idp_password_creator, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 最終承認タスクの「開く」をクリックする

最終承認フォームダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-dashboard').locator('.fa-pencil').click()
    await expect(page.locator('#workflow-field-creator_approval_result')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 最終承認フォームで「差し戻し」を選択し「コメント」を入力して「送信」をクリックする

タスクが送信され、creatorにはタスクが表示されないこと（managerに差し戻し）

In [ ]:
creator_reject_comment = 'Rejected by creator - please review again'

async def _step(page):
    option = page.locator('#workflow-field-creator_approval_result option:has-text("差し戻し")')
    value = await option.get_attribute('value')
    await page.locator('#workflow-field-creator_approval_result').select_option(value=value)
    await page.locator('#workflow-field-creator_comment').fill(creator_reject_comment)
    await page.locator('//button[@data-test-task-dialog-submit]').click()
    await expect(page.locator('//button[@data-test-task-dialog-submit]')).not_to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//button[@data-test-workflow-task-open]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## creatorをログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_creator, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクトURLを開きmanagerとしてログインする

プロジェクトダッシュボードに承認タスクの「開く」ボタンが表示されること

In [ ]:
async def _step(page):
    await asyncio.sleep(10)
    await page.goto(project_url)
    await grdm.login(page, idp_name_manager, idp_username_manager, idp_password_manager, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## コメントボタンをクリックする

コメントパネルが開き、creatorからの差し戻しコメントが表示されること

In [ ]:
async def _step(page):
    await expect(page.locator('.fa-comments-o')).to_be_visible(timeout=transition_timeout)
    await page.locator('.fa-comments-o').click()
    await expect(page.locator(f'//*[contains(text(), "{creator_reject_comment}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 承認タスクの「開く」をクリックする

審査フォームダイアログが表示されること

In [ ]:
async def _step(page):
    await page.reload()
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_be_visible(timeout=transition_timeout)
    await page.locator('#workflow-dashboard').locator('.fa-pencil').click()
    await expect(page.locator('#workflow-field-approval_result')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 審査フォームで「承認」を選択し「コメント」を入力して「送信」をクリックする

タスクが送信され、managerにはタスクが表示されないこと

In [ ]:
async def _step(page):
    option = page.locator('#workflow-field-approval_result option:has-text("承認")')
    value = await option.get_attribute('value')
    await page.locator('#workflow-field-approval_result').select_option(value=value)
    await page.locator('#workflow-field-review_comment').fill('Re-approved by manager after creator rejection')
    await page.locator('//button[@data-test-task-dialog-submit]').click()
    await expect(page.locator('//button[@data-test-task-dialog-submit]')).not_to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//button[@data-test-workflow-task-open]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## managerをログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await grdm.logout(page, idp_name_manager, transition_timeout=transition_timeout)

await run_pw(_step)

## 10秒待ち、テンプレートプロジェクトURLを開きcreatorとしてログインする

プロジェクトダッシュボードに最終承認タスクの「開く」ボタンが表示されること

In [ ]:
async def _step(page):
    await asyncio.sleep(10)
    await page.goto(template_project_url)
    await grdm.login(page, idp_name_creator, idp_username_creator, idp_password_creator, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 最終承認タスクの「開く」をクリックする

最終承認フォームダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-dashboard').locator('.fa-pencil').click()
    await expect(page.locator('#workflow-field-creator_approval_result')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 最終承認フォームで「承認」を選択し「コメント」を入力して「送信」をクリックする

ワークフローが完了し、タスク一覧にタスクが表示されないこと

In [ ]:
async def _step(page):
    option = page.locator('#workflow-field-creator_approval_result option:has-text("承認")')
    value = await option.get_attribute('value')
    await page.locator('#workflow-field-creator_approval_result').select_option(value=value)
    await page.locator('#workflow-field-creator_comment').fill('Final approval by creator')
    await page.locator('//button[@data-test-task-dialog-submit]').click()
    await expect(page.locator('//button[@data-test-task-dialog-submit]')).not_to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//button[@data-test-workflow-task-open]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## ワークフローが完了したことを確認する

ワークフローパネルにタスクが表示されないこと

In [ ]:
async def _step(page):
    await page.reload()
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## 後始末

In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)

In [ ]:
!rm -fr {work_dir}